# knot — 05: embeddings + entity resolution

Two sources have published overlapping movie records (imdb in
02_ingest, tmdb in 04_ingest_tmdb). Each row has its own
`source_identifier`; none has a `canonical_id` yet. The
resolver view is empty.

This notebook runs two independent worker stages — same
decoupling pattern as ingest:

| stage | scans for | does |
|---|---|---|
| **embedding** | `title_embedding IS NULL` | encodes title → UPDATE column |
| **ER** | `canonical_id IS NULL` | k-NN against other sources → assign canonical |

knot owns the *schema* (`VECTOR(384)` slot + HNSW index, built
by `Spec.ddl()` in 01 and extended by 03) and the *SQL* for
canonical assignment (`binding.assign_canonical_sql()`).
Everything else — encoder choice, threshold, blocking
strategy, reranker — belongs to the host worker.

In [2]:
import json
import uuid

import pandas as pd
from _demo import connect
from sentence_transformers import SentenceTransformer

/mnt/main/code/knot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Full spec composed in (we need both source bindings).
# tmdb_movie_b lives in the movies.py domain file once .full
# has been imported (it's defined there, full just composes).
import movies_spec.full  # noqa: F401
from movies_spec import imdb_movie_b, movie
from movies_spec.movies import tmdb_movie_b

In [4]:
pg, engine = connect()

In [5]:
# Resolved view is empty going in: every row's canonical_id is
# NULL, the view filters those out.
resolved_before = pd.read_sql_query(movie.resolved.sql(), engine)
print(f"resolved rows BEFORE ER: {len(resolved_before)}")
resolved_before

resolved rows BEFORE ER: 0


,canonical_id,title,year,director,runtime_minutes,title_embedding


## Stage 1 — embedding worker

Load the encoder, find unembedded rows, encode their titles in
one batch, push back via UPDATE. A real worker would loop
forever; here we do it once.

knot's contribution: the `title_embedding vector(384)` column
+ the HNSW index with `vector_cosine_ops`. The model
(`all-MiniLM-L6-v2`, 384 dims) is the worker's choice and
belongs in worker config, not knot.

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"loaded encoder · dim={model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11241.86it/s]

loaded encoder · dim=384


/tmp/ipykernel_569176/2791016111.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"loaded encoder · dim={model.get_sentence_embedding_dimension()}")


In [7]:
# Use the per-class accessor instead of f-string-baking the
# bindings table name. One place to change the suffix convention;
# the notebook stays declarative.
table = movie.bindings_table_name
unembedded = pd.read_sql_query(
    f"""
    SELECT source_name, source_identifier, valid_from, title
    FROM {table}
    WHERE title_embedding IS NULL
    """,
    engine,
)
embeddings = model.encode(unembedded["title"].tolist(), normalize_embeddings=True)
print(f"encoded {len(unembedded)} titles → {embeddings.shape}")

with pg.cursor() as cur:
    for (sn, si, vf, _), vec in zip(
        unembedded.itertuples(index=False), embeddings, strict=False
    ):
        cur.execute(
            f"""
            UPDATE {table}
            SET title_embedding = %(vec)s::vector(384)
            WHERE source_name = %(sn)s
              AND source_identifier = %(si)s
              AND valid_from = %(vf)s
            """,
            {"vec": str(vec.tolist()), "sn": sn, "si": si, "vf": vf},
        )

encoded 203 titles → (203, 384)


In [8]:
# Confirm: every row now has an embedding.
pd.read_sql_query(
    f"""
    SELECT source_name, COUNT(*) AS rows, COUNT(title_embedding) AS embedded
    FROM {movie.bindings_table_name}
    GROUP BY source_name
    ORDER BY source_name
    """,
    engine,
)

,source_name,rows,embedded
0,imdb,70,70
1,rottentomatoes,65,65
2,tmdb,68,68


## Stage 2 — ER worker (k-NN blocking + threshold)

Two-phase: mint canonical_ids for imdb (the anchor source),
then for each tmdb row find the nearest imdb row by cosine
distance and either reuse the matched canonical_id or mint a
new one if no neighbor is close enough.

The k-NN query uses `<=>` (cosine distance via
`vector_cosine_ops`) — the HNSW index does the heavy lifting,
O(log n) per lookup. The pgvector idiom for "k=1 nearest
neighbor" is `ORDER BY embedding <=> query LIMIT 1`.

This notebook keeps the policy deliberately simple: single
candidate, hard threshold, no reranker. The full cascade
(sparse blocker → dense blocker → cross-encoder reranker →
LLM on the ambiguous middle) layers on the same primitives.
knot's contract ends at `binding.assign_canonical_sql()`.

In [9]:
# Phase 1: mint canonical_id for every imdb row. imdb is the
# "anchor" — in production you'd pick the most trusted source
# or use a deterministic key.
imdb_rows = pd.read_sql_query(
    f"SELECT source_identifier, title "
    f"FROM {movie.bindings_table_name} "
    f"WHERE source_name = 'imdb' AND canonical_id IS NULL",
    engine,
)
print(f"minting {len(imdb_rows)} canonical_ids for imdb")

assign_imdb = imdb_movie_b.assign_canonical_sql()
with pg.cursor() as cur:
    for si in imdb_rows["source_identifier"]:
        cur.execute(
            assign_imdb,
            {
                "canonical_id": f"m_{uuid.uuid4().hex[:10]}",
                "source_identifier": si,
                "er_metadata": json.dumps({"method": "mint", "source": "imdb"}),
            },
        )

minting 70 canonical_ids for imdb


In [10]:
# Phase 2 (read): for each tmdb row, find the nearest imdb row.
# CROSS JOIN LATERAL drives the per-row k-NN.
bindings = movie.bindings_table_name
candidates = pd.read_sql_query(
    f"""
    SELECT
        t.source_identifier  AS tmdb_id,
        t.title              AS tmdb_title,
        i.source_identifier  AS imdb_id,
        i.title              AS imdb_title,
        i.canonical_id       AS imdb_canonical,
        (t.title_embedding <=> i.title_embedding) AS distance
    FROM {bindings} t
    CROSS JOIN LATERAL (
        SELECT source_identifier, title, canonical_id, title_embedding
        FROM {bindings}
        WHERE source_name = 'imdb' AND canonical_id IS NOT NULL
        ORDER BY title_embedding <=> t.title_embedding
        LIMIT 1
    ) i
    WHERE t.source_name = 'tmdb' AND t.canonical_id IS NULL
    ORDER BY distance
    """,
    engine,
)
candidates.head(15)

,tmdb_id,tmdb_title,imdb_id,imdb_title,imdb_canonical,distance
0,560763,Jaws,tt8421613,Jaws,m_3102a94e33,0.0
1,705177,Inglourious Basterds,tt5322801,Inglourious Basterds,m_1c08d0c3b8,0.0
2,413638,Django Unchained,tt0768738,Django Unchained,m_c5c3d28384,0.0
3,694525,Tokyo Story,tt5674915,Tokyo Story,m_c57c8099bc,0.0
4,897238,Spirited Away,tt9672202,Spirited Away,m_385bcd669a,0.0
5,369157,Howl's Moving Castle,tt0681347,Howl's Moving Castle,m_bf367a1e68,0.0
6,136783,Mirror,tt2281070,Mirror,m_ed2e64686a,0.0
7,464270,Taxi Driver,tt0451655,Taxi Driver,m_58f3e4f152,0.0
8,365590,The Departed,tt4478005,The Departed,m_706005c90f,0.0
9,128926,Persona,tt3553450,Persona,m_ef708dfed0,0.0


In [11]:
# Phase 2 (write): threshold + assign. cos distance ≤ 0.20 →
# similarity ≥ 0.80. Reasonable cut for all-MiniLM-L6-v2 on movie
# titles; production tunes this against labeled data.
THRESHOLD = 0.20
matched = int((candidates["distance"] <= THRESHOLD).sum())
print(f"matched {matched}/{len(candidates)} tmdb rows at distance ≤ {THRESHOLD}")

assign_tmdb = tmdb_movie_b.assign_canonical_sql()
with pg.cursor() as cur:
    for row in candidates.itertuples(index=False):
        if row.distance <= THRESHOLD:
            canonical_id = row.imdb_canonical
            method = "matched"
            matched_imdb_id = row.imdb_id
        else:
            canonical_id = f"m_{uuid.uuid4().hex[:10]}"
            method = "mint"
            matched_imdb_id = None
        cur.execute(
            assign_tmdb,
            {
                "canonical_id": canonical_id,
                "source_identifier": row.tmdb_id,
                "er_metadata": json.dumps(
                    {
                        "method": method,
                        "distance": float(row.distance),
                        "matched_imdb_id": matched_imdb_id,
                    }
                ),
            },
        )

matched 50/68 tmdb rows at distance ≤ 0.2


## Confirm — resolver view picks the winner per slot

The resolver argmaxes per `(canonical_id, slot)` across both
sources' weights. imdb has weight 0.85, tmdb 0.70 — so for
matched rows where both sources agree-but-disagree, imdb wins.
The `_all_sources` view exposes both claims per slot.

In [12]:
pd.read_sql_query(
    movie.resolved.order_by(movie.col.year, "desc").limit(15).sql(),
    engine,
)

,canonical_id,title,year,director,runtime_minutes,title_embedding
0,m_4d9a97d20f,Dune: Part Two,2024,p_villeneuve,167,"[-0.037842646,0.036646675,0.021372823,-0.03812..."
1,m_8cae5b916d,The Boy and the Heron,2023,p_miyazaki,124,"[-0.055139694,0.082437664,-0.072895385,-0.0436..."
2,m_7f658e0ba0,Killers of the Flower Moon,2023,p_scorsese,207,"[-0.027064158,0.014361586,0.047937084,-0.00047..."
3,m_54a3f6132d,Poor Things,2023,p_lanthimos,141,"[0.0023410106,0.07631819,0.06358255,0.06979304..."
4,m_f38b1035e8,Oppenheimer,2023,p_nolan,179,"[-0.09202981,0.02925532,-0.032360878,0.1133780..."
5,m_fbbcd9fc63,Barbie,2023,p_gerwig,114,"[0.0133914035,0.03366768,-0.004840464,0.006915..."
6,m_3224112e17,Decision to Leave,2022,p_park,140,"[0.057176586,0.037560448,0.006829723,0.0330268..."
7,m_bfdfad9504,The Northman,2022,p_eggers,136,"[-0.003956684,0.06541376,-0.015326507,-0.05164..."
8,m_07f85c642a,Dune,2021,p_villeneuve,156,"[-0.0065636984,0.022691706,-0.015095017,-0.011..."
9,m_050819597c,Pain and Glory,2019,p_almodovar,112,"[-0.021179356,0.087038666,0.017359592,-0.00443..."


In [13]:
# Per-canonical breakdown — how many sources contributed to each.
pd.read_sql_query(
    f"""
    SELECT canonical_id,
           jsonb_object_agg(source_name, source_identifier) AS sources,
           COUNT(*) AS source_count
    FROM {movie.bindings_table_name}
    WHERE canonical_id IS NOT NULL AND valid_to IS NULL
    GROUP BY canonical_id
    ORDER BY source_count DESC, canonical_id
    LIMIT 15
    """,
    engine,
)

,canonical_id,sources,source_count
0,m_00777f748f,"{'imdb': 'tt6027839', 'tmdb': '457803'}",3
1,m_031867e7b3,"{'imdb': 'tt6387155', 'tmdb': '831642'}",2
2,m_04306aec4e,"{'imdb': 'tt5235963', 'tmdb': '733477'}",2
3,m_050819597c,"{'imdb': 'tt9013509', 'tmdb': '536995'}",2
4,m_06677d31ed,"{'imdb': 'tt7447732', 'tmdb': '192219'}",2
5,m_1a3a23af00,"{'imdb': 'tt9810970', 'tmdb': '28231'}",2
6,m_1c08d0c3b8,"{'imdb': 'tt5322801', 'tmdb': '705177'}",2
7,m_20ccea09ec,"{'imdb': 'tt7960788', 'tmdb': '543164'}",2
8,m_2518edb0d7,"{'imdb': 'tt0951972', 'tmdb': '717468'}",2
9,m_2b1113ba0f,"{'imdb': 'tt4180883', 'tmdb': '588557'}",2


## Where knot's responsibility ends

| knot | worker |
|---|---|
| `VECTOR(384)` column + HNSW index in `Spec.ddl()` | picked the encoder, dim, metric |
| `binding.write_sql()` for ingest | wrote the embedding-fill UPDATE loop |
| `binding.assign_canonical_sql()` for ER stamping | picked the threshold, blocking strategy, mint scheme |
| resolver views over canonical bindings | wrote the candidate-query SQL using `<=>` |
| `er_metadata jsonb` column on bindings | populated `{"method": ..., "distance": ...}` for audit |

The candidate-generation SQL is currently hand-written
(`<=>` against `movie_bindings`). A
`movie.col.title_embedding.distance(vec)` operator on the read
substrate would let the same query be expressed via the Query
AST. That's the natural next library addition — same shape as
FK walks and aggregates, one more Expr node + one more
`compile_sql.register`. Until then, raw SQL works.